# PartitionModel: H2S alpha correction and custom partition models

This demo shows the three things the `PartitionModel` protocol and
`HenryEquilibrium` dataclass enable:

1. **Inspect stock partition models** -- access `AD_BASIC`'s built-in
   H2S, CO2, CH4, H2, and NH3 `HenryEquilibrium` objects.
2. **Temperature dependence** -- `HenryEquilibrium` carries a van 't Hoff
   coefficient so kH varies with temperature, unlike the old pinned constants
   in the fermenter builder.
3. **H2S alpha correction** -- at pH ~= pKa (7.0) half of dissolved sulfide
   is in the non-volatile HS- form. Without the alpha correction the link
   overestimates the gas-phase H2S fraction. This demo quantifies that error
   across the AD operating pH range.
4. **Custom partition model** -- extend a database with a user-declared
   `HenryEquilibrium` for a species not in the stock databases.

## 1. Stock partition models in AD_BASIC

In [1]:
import math

from PyOMES.chemistry import HenryEquilibrium, PartitionModel
from PyOMES.databases.anaerobic_digestion import AD_BASIC

print("=== 1. Partition models in AD_BASIC ===")
for sp_id, model in sorted(AD_BASIC.partition_models.items()):
    kH_25 = model._kH_mol_L_atm(298.15)
    print(f"  {sp_id:5s}  kH(25 C) = {kH_25:.3e} mol/(L.atm)  dlnH = {model.dlnH:.0f} K")

=== 1. Partition models in AD_BASIC ===
  CH4    kH(25 C) = 1.419e-03 mol/(L.atm)  dlnH = 1600 K
  CO2    kH(25 C) = 3.445e-02 mol/(L.atm)  dlnH = 2400 K
  H2     kH(25 C) = 7.802e-04 mol/(L.atm)  dlnH = 500 K
  H2S    kH(25 C) = 1.013e-01 mol/(L.atm)  dlnH = 2100 K
  N2     kH(25 C) = 6.485e-04 mol/(L.atm)  dlnH = 1300 K
  NH3    kH(25 C) = 5.978e+01 mol/(L.atm)  dlnH = 4200 K
  O2     kH(25 C) = 1.317e-03 mol/(L.atm)  dlnH = 1500 K


## 2. Temperature dependence

In [2]:
print("=== 2. H2S temperature dependence ===")
h2s = AD_BASIC.partition_models["H2S"]
for T_C in (15, 25, 35, 45):
    T_K = T_C + 273.15
    kH = h2s._kH_mol_L_atm(T_K)
    print(f"  T = {T_C:2d} C   kH = {kH:.4f} mol/(L.atm)")

=== 2. H2S temperature dependence ===
  T = 15 C   kH = 0.1294 mol/(L.atm)
  T = 25 C   kH = 0.1013 mol/(L.atm)
  T = 35 C   kH = 0.0806 mol/(L.atm)
  T = 45 C   kH = 0.0651 mol/(L.atm)


## 3. H2S alpha correction across pH

At pH = pKa (7.0) half of total dissolved sulfide is H2S (volatile) and half
is HS- (non-volatile). `alpha = [H2S] / ([H2S] + [HS-])`.

For a monoprotic acid: `alpha = 1 / (1 + 10^(pH - pKa))`.

The gas-liquid link passes `alpha` to `HenryEquilibrium.partition_ratio()`
which divides kH by alpha, making the effective Henry constant larger (more
liquid-favoured) at higher pH. For CO2 (pKa1 = 6.35) the same correction
applies but the magnitude is small because kH is already large.

In [3]:
PKA_H2S = 7.0
T_K = 308.15   # 35C -- typical AD operating temperature
V_liq = 1.0    # L (normalised)
V_gas = 0.25   # L (headspace fraction 20% of 1.25 L total)

print("=== 3. H2S alpha correction at 35 C ===")
print(f"  pKa(H2S) = {PKA_H2S},  V_liq = {V_liq} L,  V_gas = {V_gas} L")
print()
print(f"  {'pH':>5}  {'alpha':>7}  {'f_gas (alpha=1)':>16}  {'f_gas (corrected)':>18}  {'error':>8}")
print(f"  {'-'*5}  {'-'*7}  {'-'*16}  {'-'*18}  {'-'*8}")

for pH in (5.0, 6.0, 7.0, 7.5, 8.0, 9.0):
    alpha = 1.0 / (1.0 + 10 ** (pH - PKA_H2S))

    # Without correction: alpha fixed at 1.0
    beta_uncorrected = h2s.partition_ratio(V_liq, V_gas, T_K, alpha=1.0)
    f_gas_uncorrected = 1.0 - beta_uncorrected / (1.0 + beta_uncorrected)

    # With alpha correction
    beta_corrected = h2s.partition_ratio(V_liq, V_gas, T_K, alpha=alpha)
    f_gas_corrected = 1.0 - beta_corrected / (1.0 + beta_corrected)

    if f_gas_corrected > 0:
        error = (f_gas_uncorrected - f_gas_corrected) / f_gas_corrected
    else:
        error = float("inf")

    print(f"  {pH:5.1f}  {alpha:7.3f}  {f_gas_uncorrected:16.4f}  {f_gas_corrected:18.4f}  {error:+8.1%}")

print()
print("  Interpretation: at pH = pKa (7.0) the uncorrected model overestimates")
print("  the gas-phase H2S fraction by ~2x.  At pH 8 the error exceeds 4x.")

=== 3. H2S alpha correction at 35 C ===
  pKa(H2S) = 7.0,  V_liq = 1.0 L,  V_gas = 0.25 L

     pH    alpha   f_gas (alpha=1)   f_gas (corrected)     error
  -----  -------  ----------------  ------------------  --------
    5.0    0.990            0.1092              0.1083     +0.9%
    6.0    0.909            0.1092              0.1003     +8.9%
    7.0    0.500            0.1092              0.0578    +89.1%
    7.5    0.240            0.1092              0.0286   +281.7%
    8.0    0.091            0.1092              0.0110   +890.8%
    9.0    0.010            0.1092              0.0012  +8907.6%

  Interpretation: at pH = pKa (7.0) the uncorrected model overestimates
  the gas-phase H2S fraction by ~2x.  At pH 8 the error exceeds 4x.


## 4. Extend a database with a custom partition model

Ethanol: moderately volatile, no ionisation (no alpha correction needed).
Sander (2015) kH0 ~= 192 mol/(L.atm) at 25C, dlnH ~= 6600 K.

In [4]:
print("=== 4. Custom partition model ===")

EtOH_PARTITION = HenryEquilibrium(H_ref=1.9e0, dlnH=6600.0)

MY_DB = AD_BASIC.extend(partition_models={"Ethanol": EtOH_PARTITION})
print(f"  AD_BASIC partition_models : {sorted(AD_BASIC.partition_models)}")
print(f"  MY_DB   partition_models  : {sorted(MY_DB.partition_models)}")
print()

kH_etoh_25 = MY_DB.partition_models["Ethanol"]._kH_mol_L_atm(298.15)
kH_etoh_35 = MY_DB.partition_models["Ethanol"]._kH_mol_L_atm(308.15)
print(f"  Ethanol kH(25 C) = {kH_etoh_25:.1f} mol/(L.atm)")
print(f"  Ethanol kH(35 C) = {kH_etoh_35:.1f} mol/(L.atm)")
print()

# Protocol check: HenryEquilibrium satisfies PartitionModel
assert callable(MY_DB.partition_models["Ethanol"].partition_ratio)
assert callable(MY_DB.partition_models["Ethanol"].equilibrium_a_moles)
print("  HenryEquilibrium satisfies PartitionModel protocol: OK")
print()

print("All assertions passed. PartitionModel demo complete.")

=== 4. Custom partition model ===
  AD_BASIC partition_models : ['CH4', 'CO2', 'H2', 'H2S', 'N2', 'NH3', 'O2']
  MY_DB   partition_models  : ['CH4', 'CO2', 'Ethanol', 'H2', 'H2S', 'N2', 'NH3', 'O2']

  Ethanol kH(25 C) = 192.5 mol/(L.atm)
  Ethanol kH(35 C) = 93.9 mol/(L.atm)

  HenryEquilibrium satisfies PartitionModel protocol: OK

All assertions passed. PartitionModel demo complete.
